# RAG 第 21 课：企业级 RAG on AWS（架构篇，不写代码）

前 20 课我们把一个 RAG 系统该有的所有算法组件都打磨过了——chunking、BM25、dense、hybrid、rerank、HyDE、compression、citations、multi-turn、ablation。

但真正上线到公司生产环境时，你面对的问题是**完全不同的另一类**：

- 10 万份文档每天新增 500 份，怎么自动入库？
- 多租户，A 公司数据不能被 B 公司检索到，怎么隔离？
- 谁看了哪份文档、生成了什么答案，审计日志怎么留？
- 延迟要 P95 < 2 秒，成本要可控，挂了要能恢复——这些由谁保证？

这一课我们不写代码，只聊**架构**：假设你在 AWS 上从零搭建一个企业级 RAG，每一块选什么服务、为什么。

> 注：这里不是 AWS 官方推荐架构，是我基于你熟悉的 AWS 技术栈给出的一个**工程参考方案**。真实项目会根据业务量级、合规要求、预算再调整。

## 1. 把 RAG 系统拆成四条管线

不管多复杂，企业级 RAG 都能拆成这四条独立的管线（pipeline）：

```
     ┌──────────────────────────────────────────────────────────────┐
     │  A. Ingestion Pipeline （离线，T+N 运行）                      │
     │     文件 → 解析 → 切块 → 向量化 → 写入向量库 + 元数据库        │
     └──────────────────────────────────────────────────────────────┘

     ┌──────────────────────────────────────────────────────────────┐
     │  B. Query Pipeline （在线，毫秒级）                            │
     │     用户问 → 改写 → 检索 → rerank → compress → LLM → 引用     │
     └──────────────────────────────────────────────────────────────┘

     ┌──────────────────────────────────────────────────────────────┐
     │  C. Evaluation Pipeline （离线 + 灰度）                        │
     │     gold set → 跑 pipeline → 算 Hit@k / F1 / 人工打分 → 看板  │
     └──────────────────────────────────────────────────────────────┘

     ┌──────────────────────────────────────────────────────────────┐
     │  D. Observability & Governance （贯穿全流程）                  │
     │     日志 / trace / 成本 / guardrails / 权限审计              │
     └──────────────────────────────────────────────────────────────┘
```

这四条管线**部署节奏、SLA、所属团队**往往都不一样。架构设计的第一刀就是把它们拆开。

## 2. Ingestion Pipeline：文档怎么进来

**目标**：从公司各种源头（SharePoint、Confluence、S3、RDS、邮件归档……）把文档拿进来，解析、切块、向量化，落地到向量库。

### 典型 AWS 组件选型

| 环节 | 服务 | 为什么 |
|---|---|---|
| 原始文件存放 | **S3**（+ S3 Event Notification） | 便宜、事件驱动、版本化、跨区复制 |
| 触发器 | **EventBridge** 或 **S3 → Lambda** | 新文件落地立刻触发入库 |
| 解析（PDF/Office/HTML） | **Textract**（PDF 扫描件）、**Lambda** 跑 `unstructured` / `pypdf` | Textract 专门擅长扫描件和表格 |
| 切块 + 清洗 | **Lambda** 或 **AWS Glue / EMR**（大批量） | 小量用 Lambda，TB 级用 Glue |
| 向量化 | **Bedrock: Titan Embeddings v2** 或 **Cohere Embed Multilingual** | 托管、按量付费、不用自己维护 GPU |
| 工作流编排 | **Step Functions** | 失败重试、分支、可视化 |
| 向量库 | **OpenSearch Serverless** 向量引擎 / **Aurora PostgreSQL + pgvector** / **Bedrock Knowledge Bases** 托管的底层 | 见下一节 |
| 元数据库（doc 来源、权限、版本） | **DynamoDB** 或 **Aurora** | DynamoDB 适合大规模、简单查询 |

### 工程要点

1. **幂等性**：同一份文档重新入库不能产生两套 chunk。用 `sha256(doc_content)` 作为业务主键。
2. **增量更新**：S3 版本号变了 → 把旧 chunk 的向量打 `deleted=true`，再写入新 chunk。**不要真删**，审计要看。
3. **死信队列 (DLQ)**：解析失败的文档不能静默丢掉，扔到 SQS DLQ 人工看。
4. **分层切块**：一份文档同时产出 "章节级大 chunk" + "段落级小 chunk"，查询时可以先粗后细（课堂第 11 课的 small-to-big 思想）。

## 3. 向量库怎么选：三条主流路线

这是企业 RAG on AWS 最容易踩坑的选型。不同方案的"适合谁"很不一样。

### 路线 A：Bedrock Knowledge Bases（全托管）

- **是什么**：你把文档扔 S3，配好 KB，AWS 帮你做 parsing / chunking / embedding / 向量库 / 检索。底层默认用 OpenSearch Serverless（也可挂 Aurora pgvector / Pinecone）。
- **适合**：团队小、想尽快上线、不需要极致定制。
- **放弃**：chunking 策略、rerank、混合检索细节基本不可调（**正在快速迭代，能力在扩**）。
- **配套**：Bedrock Agents 可以直接调用 KB，做 Agentic RAG。

### 路线 B：OpenSearch（自建/Serverless） + 自研 pipeline

- **是什么**：向量库用 OpenSearch 的 `knn_vector` 字段，BM25 本身 OpenSearch 原生支持（它就是 Lucene）。**一个库搞定混合检索**。
- **适合**：想做我们课上讲的完整 hybrid + rerank + 自定义 chunking 流程。
- **优势**：原生 k-NN（HNSW / IVF）+ 原生 BM25 + 原生 filter（多租户 / 权限过滤） + SQL / DSL 都能用。
- **成本**：Serverless 按用量，小流量可能不划算；大流量适合。

### 路线 C：Aurora PostgreSQL + pgvector

- **是什么**：向量直接存在关系库里，和业务数据共用事务。
- **适合**：你已经重度用 Aurora，文档数量在百万级以下，对"强一致 + 事务"要求高（比如文档和权限表必须同一个事务写）。
- **放弃**：到几千万向量时，查询延迟和索引维护会比 OpenSearch 难。

### 我的经验法则

- **MVP / 内部工具** → Bedrock Knowledge Bases。1 天能上线。
- **产品化、需要定制** → OpenSearch。本课 14 课的 hybrid 能原生跑。
- **强事务 / 少量数据** → Aurora pgvector。
- 不是非黑即白，真实项目里常见的是"**冷数据 OpenSearch + 热业务数据 Aurora + 对外统一网关**"这种组合。

## 4. Query Pipeline：在线链路怎么搭

用户问一句话，从进来到拿到带引用的答案，走过的路：

```
User
 ├─ Cognito / IAM Identity Center      认证 + 拿到 userId / tenantId
 ├─ CloudFront + WAF                   防刷、限流、地域加速
 ├─ API Gateway (REST / WebSocket)     入口 + throttling
 └─ Lambda (或 ECS Fargate)            业务编排
       │
       ├─ 1) Query Reformulation     Bedrock Claude Haiku（快、便宜）
       ├─ 2) Retrieval               OpenSearch hybrid（BM25 + kNN）
       │                             filter: tenantId / acl / lang
       ├─ 3) Rerank                  Bedrock Cohere Rerank 3
       ├─ 4) Compression (可选)      Claude Haiku 提取相关句
       ├─ 5) Answer + Citations      Bedrock Claude Sonnet / Opus
       ├─ 6) Grounding Check         Haiku judge + lexical
       └─ 7) Guardrails              Bedrock Guardrails（过滤敏感输出）
 └─ 返回给前端（SSE 流式）
```

### 几个关键取舍

1. **为什么 Lambda 而不是一直开着的 ECS？** 如果 QPS 可预测（比如内部工具夜间 0 流量），Lambda 省钱；QPS 高且稳定，Fargate / EKS 每请求成本更低。
2. **为什么用流式 SSE 返回？** LLM 第一个 token 到最后一个 token 差几秒，用户体验区别巨大。API Gateway WebSocket 或 Lambda Response Streaming 都能做。
3. **大模型分层**：别所有步骤都用 Claude Opus。**改写、判别、grounding 用 Haiku**（便宜 20 倍、快 3 倍），**最终生成用 Sonnet/Opus**。你会发现 80% 成本来自最后那一步。
4. **缓存**：同一个问题 10 分钟内再问，没必要重算。用 **ElastiCache (Redis)** 按 `hash(tenantId + normalized_question)` 缓存最终答案。命中直接返回，**同时仍然记录日志**（不能让缓存绕过审计）。

## 5. 多租户 & 权限：企业 RAG 最容易翻车的点

**黄金规则：权限过滤必须发生在检索层，不能靠 LLM 过滤。**

LLM 不是安全边界。你不能在 prompt 里写 "请不要把 B 公司的内容告诉 A 公司用户"——它做不到。

正确的做法：

1. 每个 chunk 入库时带 metadata：`tenantId`、`acl_groups: [...]`、`classification: 'internal'|'confidential'|...`
2. 在线查询时，从 **Cognito / IAM Identity Center** 拿到当前用户的身份，解析出 `tenantId` 和所属 group 列表
3. 向 OpenSearch 发起检索时，**用 filter 强制圈定范围**：
   ```
   { "bool": {
       "must":   [ kNN / BM25 query ],
       "filter": [
         { "term":  { "tenantId": "<current>" }},
         { "terms": { "acl_groups": [<user groups>] }}
       ]
   }}
   ```
4. 拿到 doc_id 后，**再做一次** "这个用户对这个 doc 是否可读" 的二次校验（防止 stale metadata）
5. 最终答案生成时，记录 `userId + docIds used` 到审计日志（CloudTrail / S3 + Athena 查询）

很多团队第一次上线只做了第 3 步，**漏了第 4 步**——新员工被从某个 group 踢出后，缓存/索引没立即同步，导致泄漏。

## 6. 安全 & 合规

企业买单往往不是因为 RAG 答得准，而是**因为安全过关**。

- **静态加密**：S3 / OpenSearch / Aurora / DynamoDB 全部开 **KMS CMK**（客户管理密钥），不要用 AWS 默认 key——合规审计时过不了。
- **传输加密**：所有服务间 TLS 1.2+，内部链路走 **VPC Endpoints / PrivateLink**，不出公网。
- **Bedrock 数据不出 region**：在 VPC Endpoint 里调 Bedrock，input/output 不经互联网。
- **零保留**：Bedrock 默认不用你的数据训练，但你可以额外签 DPA 并且**关闭 model invocation logging** 的用户输入部分，只保留 metadata（谁调了、什么时候），不保留 prompt 原文。
- **PII 处理**：入库前用 **Amazon Comprehend** 或 **Macie** 扫描，敏感字段 mask 掉。或者做 **tokenization**（替换为占位符，在 LLM 生成后再替换回来）。
- **Bedrock Guardrails**：输出侧拦截敏感话题、PII 泄漏、越狱 prompt。比你自己在 system prompt 里写 "请不要说脏话" 稳得多。
- **审计**：CloudTrail 记录所有 API 调用；应用层日志（谁问了什么、用了哪些 doc、生成了什么答案）落 S3，用 **Athena / QuickSight** 查询。
- **合规证书**：如果你的客户要 SOC2 / HIPAA / PCI / FedRAMP，走 AWS 已认证服务的组合会省掉 6 个月的审计工作。

## 7. 可观测性（Observability）

一个 RAG 系统上线后最容易被问的三个问题：

1. "今天为什么变慢了？" → 需要 **latency 分解**
2. "这次答错是哪一步错了？" → 需要 **trace**
3. "这个月花了多少钱？" → 需要 **成本归因**

### 推荐组合

- **Metrics**：CloudWatch Metrics（每一步 latency、token 数、cache hit rate、retrieval hit@k）
- **Tracing**：**AWS X-Ray** 或 **OpenTelemetry → ADOT**，给每个请求一个 trace_id，串起 Lambda / OpenSearch / Bedrock 所有 span
- **Logs**：结构化 JSON 写 CloudWatch Logs → **Firehose → S3**，Athena 查询
- **LLM 专用可观测**：第三方工具如 **Langfuse / Arize / Phoenix** 可托管在 ECS 里，专门看 prompt / response / token / 评分。这一层是 CloudWatch 做不到的，因为它看不懂 prompt 语义。
- **Cost**：Bedrock 的 invocation 可以打 tag，按 tag 进 **Cost Explorer** 分账；OpenSearch 按 collection 分账。**规定每个团队必须打 `team` tag**，否则三个月后账单没人看得懂。

### 一定要追的三个指标

1. **Retrieval Hit@k**：检索质量。gold set 离线跑。下降就是索引出问题了。
2. **Grounding Rate**（第 19 课的指标）：生成的句子有多少能被引用文档支持。下降意味着模型在瞎编。
3. **P95 Latency 按环节分解**：检索 / rerank / LLM 三段独立看。慢是哪一段慢，一眼看出来。

## 8. 成本：企业 RAG 的真实账单长什么样

经验比例（**中等规模，100 万 chunks + 5 万 日请求**）：

| 项 | 占比 | 说明 |
|---|---|---|
| LLM 生成（Bedrock Claude） | **50–70%** | 最终答案那一步是大头 |
| Embedding（离线 + 在线 query 向量化） | 5–10% | 入库时一次性，之后只算查询侧 |
| 向量库（OpenSearch Serverless OCU） | 10–20% | 按 OCU 小时计费，不管你用不用 |
| Rerank / judge（Haiku） | 5–10% | 便宜但量大 |
| 网络 / 存储 / 日志 | 5–10% | |

### 真正能省钱的几招（按收益排序）

1. **给生成模型做分级**：80% 问题用 Sonnet，明确复杂的才升 Opus。可以用 "长度 + 关键词" 做轻量路由。
2. **Prompt caching**（Bedrock 已支持）：system prompt / 长文档部分缓存，重复调用跳过输入 token 计费。对 RAG 的长 context 非常有效。
3. **结果缓存**：第 4 节说过。命中率 20–30% 是常见的。
4. **压缩 context**（第 17 课）：把 top-10 文档压到 3–5 段，**输入 token 直接少一半**。
5. **冷热分层**：90 天没查过的 chunk 迁到便宜的 tier（比如 OpenSearch UltraWarm / S3 + Athena），查询路径只打热层。

## 9. 上线节奏建议（给你一个 6 周路线图）

如果你明天就要启动，可以按这个节奏：

**Week 1 — Baseline**
- 用 **Bedrock Knowledge Bases** + S3，30 分钟跑通 demo
- 跑课上第 10 课的 gold-set 评测，拿 baseline 指标

**Week 2 — 切到可控架构**
- 换成 **OpenSearch Serverless + 自己的 Lambda pipeline**，跑通 hybrid 检索
- Ingestion 用 **Step Functions** 编排

**Week 3 — 质量**
- 加 rerank (Cohere Rerank 3 on Bedrock) + compression
- 跑 ablation（课上第 20 课），确认每个组件真的加了分才保留

**Week 4 — 多租户 + 权限**
- Cognito + metadata filter，把本文档第 5 节的流程跑通
- 加审计日志

**Week 5 — 生产化**
- Guardrails、KMS、VPC Endpoints、WAF、throttling
- CloudWatch dashboards + X-Ray traces
- 灰度：5% → 25% → 100%

**Week 6 — 优化**
- 结果缓存 + prompt caching
- 模型分级路由
- 按账单反向优化占比最大的环节

## 10. 工程直觉总结

1. **架构拆成四条管线**：ingestion / query / eval / observability。它们发布节奏不同，团队也不同，别耦合。
2. **向量库三选一**：Bedrock KB（快上线）/ OpenSearch（要定制）/ Aurora pgvector（强事务）。按量级和需求定。
3. **LLM 不是安全边界**。权限过滤必须在检索层 filter，不能指望 prompt。
4. **模型分级是省钱第一招**。Haiku 干杂活，Sonnet/Opus 写终稿。
5. **Prompt caching + 结果缓存** 合起来能省 30–40%。
6. **一开始就要埋观测**：trace_id、每步 latency、每步 token、grounding rate。线上出了问题没 trace 就是抓瞎。
7. **灰度发布 + Ablation 思维永远在线**：改动一个参数就跑一次评测，回答"改完到底是变好了还是变差了"，不要凭感觉。

---

## 这门课到这里结束了

一路走下来：

- 1–10 课：RAG 的算法地基（chunking / retrieval / 评测）
- 11–17 课：质量提升的各种武器（hybrid / rewrite / HyDE / compression）
- 18–20 课：产品化特性（多轮 / citations / ablation）
- **21 课（本课）：企业落地的架构蓝图**

之后想扩展的话，几个自然方向：

- **Agentic RAG**：Bedrock Agents + tool use，让 LLM 自己决定查几次、查哪里
- **Multimodal RAG**：表格、图片、PDF 里的图（Textract + Titan Multimodal Embedding）
- **Graph RAG**：把文档里的实体关系建成 Neptune 图，检索走图查询
- **Fine-tuning vs RAG**：什么时候该微调而不是加检索，Bedrock Custom Models 怎么配合

这些都可以按你的兴趣继续拓课。